# 基于 Neo4j 与大模型的天气知识图谱构建

从 CSV 天气数据出发，通过大模型抽取知识，导入 Neo4j 图数据库进行存储和可视化。

---

## 1. 项目整体架构

```
CSV 天气数据
     │
     ├──→ Protege（OWL 本体）──→ SPARQL 查询
     │
     ├──→ 大模型（知识抽取）──→ 实体 / 关系 / 规律
     │        ├── DeepSeek-V3
     │        ├── 通义千问-Max
     │        └── MiMo-v2.5-pro
     │
     └──→ Neo4j（图数据库）──→ 可视化图谱
```

**技术栈**：Python + OpenAI SDK + Neo4j + Protege

---
## 2. 环境准备

### 2.1 安装依赖

In [ ]:
# 安装所需 Python 包
!pip install openai neo4j python-dotenv rdflib -q

In [ ]:
import csv
import json
import os
import time
import re
from openai import OpenAI
from neo4j import GraphDatabase
from dotenv import load_dotenv
from IPython.display import display, Markdown

load_dotenv()
print("依赖加载完成")

### 2.2 配置 API Key

在项目目录下创建 `.env` 文件，内容如下：

```env
DEEPSEEK_API_KEY=sk-xxxx
DEEPSEEK_BASE_URL=https://api.deepseek.com/v1

QWEN_API_KEY=sk-xxxx
QWEN_BASE_URL=https://dashscope.aliyuncs.com/compatible-mode/v1

MIMO_API_KEY=sk-xxxx
MIMO_BASE_URL=https://api.xiaomimimo.com/v1

NEO4J_URI=bolt://localhost:7687
NEO4J_USER=neo4j
NEO4J_PASSWORD=your_password
```

### 2.3 Neo4j 安装与配置

**Neo4j** 是图数据库，用节点和关系存储数据（像蜘蛛网，不像表格）。

| 端口 | 用途 |
|------|------|
| 7474 | HTTP 浏览器界面（Neo4j Browser） |
| 7687 | Bolt 协议，程序连接用 |

**安装步骤**：
1. 下载 [Neo4j Desktop](https://neo4j.com/download/) 并安装
2. 创建本地 DBMS，设置密码
3. 启动数据库，状态变绿（Running）
4. 浏览器访问 `http://localhost:7474`

---
## 3. 数据探索

In [ ]:
CSV_FILE = "weather_data.csv"

# 读取 CSV 并查看基本信息
with open(CSV_FILE, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    rows = list(reader)

print(f"总记录数: {len(rows)}")
print(f"字段: {list(rows[0].keys())}")
print(f"时间范围: {rows[0]['date']} ~ {rows[-1]['date']}")
print(f"\n前 3 行数据:")
for r in rows[:3]:
    print(f"  {r['date']}  最高温:{r['temperature_2m_max']}  降水:{r['precipitation_sum']}  干旱:{r['dry_day']}")

---
## 4. 大模型知识抽取

### 4.1 方案设计

**抽取目标**（大模型从 CSV 样本中归纳）：
- **实体**：天气事件（干旱、高温、降雨）、时间段、地点
- **关系**：干旱→之后→降雨、高温伴随干旱
- **规律**：统计性气象规律（如"干旱10天后雨量较大"）

**为什么只取 50 行样本**：大模型的任务不是逐条导入数据，而是**发现规律**。看 50 行和看 1097 行总结出的规律差不多，但成本差 20 倍。

| 对比模型 | 说明 |
|---------|------|
| DeepSeek-V3 | 国产高性能模型 |
| 通义千问-Max | 阿里云旗舰模型 |
| MiMo-v2.5-pro | 小米大模型 |

### 4.2 统一 Prompt 模板

In [ ]:
SYSTEM_PROMPT = """你是一个气象知识图谱专家。你的任务是从天气数据中抽取结构化知识。

请从给定的天气 CSV 数据中抽取以下内容，以 JSON 格式返回：

1. **entities** (实体列表): 每个实体包含
   - id: 唯一标识
   - type: 实体类型 (WeatherRecord / WeatherEvent / Location / TimePeriod)
   - name: 实体名称
   - properties: 属性字典

2. **relationships** (关系列表): 每个关系包含
   - source: 源实体 id
   - target: 目标实体 id
   - type: 关系类型 (如 OCCURRED_ON, HAS_EVENT, FOLLOWED_BY, SIMILAR_TO)
   - properties: 关系属性

3. **rules** (气象规律): 从数据中发现的有意义的规律

请返回纯 JSON，不要包含 markdown 代码块标记。"""

print("Prompt 模板定义完成")

### 4.3 JSON 修复函数

大模型输出的 JSON 经常不规范，需要多层修复：
1. 去除 markdown 代码块标记
2. 修复尾部逗号（trailing comma）
3. 裸值如 `1-5` 替换为字符串
4. 补全截断的括号

In [ ]:
def parse_json_response(text):
    """从模型返回中提取 JSON，支持截断修复"""
    text = text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1]
    if text.endswith("```"):
        text = text.rsplit("```", 1)[0]
    text = text.strip()

    # 第一次：直接解析
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # 修复常见 LLM JSON 错误
    fixed = text
    fixed = re.sub(r',\s*([}\]])', r'\1', fixed)           # 去尾部逗号
    fixed = re.sub(r':\s*(-?\d+[\-~+]\d+)\s*([,}\n])', r': "\1"\2', fixed)  # 裸值->字符串
    fixed = fixed.replace("'", '"')                         # 单引号->双引号
    fixed = re.sub(r'//[^\n]*', '', fixed)                  # 去注释

    try:
        return json.loads(fixed)
    except json.JSONDecodeError:
        pass

    # 截断修复
    last_comma = fixed.rfind(",")
    last_brace = fixed.rfind("}")
    if last_comma > last_brace:
        fixed = fixed[:last_comma]
    open_braces = fixed.count("{") - fixed.count("}")
    open_brackets = fixed.count("[") - fixed.count("]")
    fixed += "]" * max(0, open_brackets) + "}" * max(0, open_braces)

    try:
        parsed = json.loads(fixed)
        print("  (JSON 已自动修复)")
        return parsed
    except json.JSONDecodeError as e:
        print(f"  JSON 解析失败: {e}")
        return None

print("JSON 修复函数定义完成")

### 4.4 调用三个大模型进行抽取

In [ ]:
# 大模型配置
MODELS = {
    "deepseek": {
        "name": "DeepSeek-V3",
        "api_key": os.getenv("DEEPSEEK_API_KEY", ""),
        "base_url": os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com/v1"),
        "model": "deepseek-chat",
    },
    "qwen": {
        "name": "通义千问-Max",
        "api_key": os.getenv("QWEN_API_KEY", ""),
        "base_url": os.getenv("QWEN_BASE_URL", "https://dashscope.aliyuncs.com/compatible-mode/v1"),
        "model": "qwen-max",
    },
    "mimo": {
        "name": "MiMo",
        "api_key": os.getenv("MIMO_API_KEY", ""),
        "base_url": os.getenv("MIMO_BASE_URL", "https://api.xiaomimimo.com/v1"),
        "model": "mimo-v2.5-pro",  # 注意：必须小写
    },
}

# 准备样本数据（前 50 行）
sample_rows = rows[:50]
sample_csv = "\n".join([
    ",".join(sample_rows[0].keys()),
    *[",".join(row.values()) for row in sample_rows[:20]],
])

user_prompt = f"""以下是天气数据的基本信息：
数据总量: {len(rows)} 条记录
时间范围: {rows[0]['date']} ~ {rows[-1]['date']}
字段: {', '.join(rows[0].keys())}

前 20 行数据样本：
{sample_csv}

请从这些数据中抽取知识图谱的实体、关系和气象规律。"""

print("模型配置和样本数据准备完成")

In [ ]:
def call_llm(config, user_prompt):
    """调用大模型 API"""
    client = OpenAI(api_key=config["api_key"], base_url=config["base_url"])
    print(f"  调用 {config['name']} ...")
    start = time.time()
    response = client.chat.completions.create(
        model=config["model"],
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.3,
        max_tokens=8000,
    )
    elapsed = time.time() - start
    content = response.choices[0].message.content
    print(f"  {config['name']} 完成，耗时 {elapsed:.1f}s，"
          f"输入 tokens: {response.usage.prompt_tokens}, "
          f"输出 tokens: {response.usage.completion_tokens}")
    return content, elapsed, response.usage

print("LLM 调用函数定义完成")

In [ ]:
# 逐个调用大模型
results = {}
for key, config in MODELS.items():
    if not config["api_key"]:
        print(f"跳过 {config['name']}: 未配置 API Key")
        continue

    print(f"\n{'='*50}")
    print(f"抽取模型: {config['name']}")
    print(f"{'='*50}")

    raw_text, elapsed, usage = call_llm(config, user_prompt)
    parsed = parse_json_response(raw_text)

    output = {
        "model": config["name"],
        "elapsed_seconds": elapsed,
        "input_tokens": usage.prompt_tokens,
        "output_tokens": usage.completion_tokens,
        "raw_response": raw_text,
        "parsed": parsed,
    }

    output_file = f"knowledge_{key}.json"
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False, indent=2)
    print(f"  结果已保存: {output_file}")

    results[key] = output

### 4.5 大模型抽取结果对比

In [ ]:
# 对比表格
print(f"{'指标':<20} | ", end="")
for key in results:
    print(f"{results[key]['model']:<20} | ", end="")
print()
print("-" * 80)

metrics = [
    ("耗时(秒)", lambda r: f"{r['elapsed_seconds']:.1f}"),
    ("输入tokens", lambda r: str(r['input_tokens'])),
    ("输出tokens", lambda r: str(r['output_tokens'])),
    ("实体数量", lambda r: str(len(r['parsed'].get('entities', []))) if r['parsed'] else "失败"),
    ("关系数量", lambda r: str(len(r['parsed'].get('relationships', []))) if r['parsed'] else "失败"),
    ("规律数量", lambda r: str(len(r['parsed'].get('rules', []))) if r['parsed'] else "失败"),
]

for name, fn in metrics:
    print(f"{name:<20} | ", end="")
    for key in results:
        print(f"{fn(results[key]):<20} | ", end="")
    print()

### 4.6 查看 DeepSeek 抽取的知识示例

In [ ]:
# 展示 DeepSeek 抽取的实体、关系和规律
ds = results.get("deepseek", {}).get("parsed", {})
if ds:
    print("=== 实体 ===")
    for e in ds.get("entities", []):
        print(f"  {e['type']}: {e['name']}")

    print("\n=== 关系 ===")
    for r in ds.get("relationships", []):
        print(f"  {r['source']} --[{r['type']}]--> {r['target']}")

    print("\n=== 规律 ===")
    for rule in ds.get("rules", []):
        if isinstance(rule, dict):
            print(f"  - {rule.get('description', str(rule))}")
        else:
            print(f"  - {rule}")

---
## 5. Neo4j 数据导入

### 5.1 图模型设计

| 节点类型 | 说明 | 来源 |
|---------|------|------|
| WeatherRecord | 每日天气记录 | CSV |
| WeatherEvent | 天气事件 | CSV + 大模型 |
| Month | 月份聚合 | 自动聚合 |
| Rule | 气象规律 | 大模型 |
| TimePeriod | 时间段 | 大模型 |

| 关系类型 | 说明 |
|---------|------|
| HAS_EVENT | 记录包含天气事件 |
| NEXT_DAY | 时间序列（前一天→后一天） |
| IN_MONTH | 记录属于某月 |
| FOLLOWED_BY | 事件先后顺序 |
| SIMILAR_TO | 相似天气模式 |

### 5.2 连接 Neo4j

In [ ]:
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

# 测试连接
with driver.session() as session:
    result = session.run("RETURN 1 AS test")
    print(f"Neo4j 连接成功: {NEO4J_URI}")

### 5.3 清空数据库并创建约束

In [ ]:
with driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")
    print("已清空数据库")

    constraints = [
        "CREATE CONSTRAINT IF NOT EXISTS FOR (r:WeatherRecord) REQUIRE r.date IS UNIQUE",
        "CREATE CONSTRAINT IF NOT EXISTS FOR (e:WeatherEvent) REQUIRE e.name IS UNIQUE",
        "CREATE INDEX IF NOT EXISTS FOR (r:WeatherRecord) ON (r.maxTemp)",
        "CREATE INDEX IF NOT EXISTS FOR (r:WeatherRecord) ON (r.month)",
    ]
    for c in constraints:
        session.run(c)
    print("已创建约束和索引")

### 5.4 导入天气记录（全部 1097 条）

**注意**：这里导入的是**全部 CSV 数据**，不是只导入 50 行样本。

大模型只看了 50 行来**发现规律**，但原始数据**全部导入** Neo4j。

In [ ]:
BOOLEAN_COLUMNS = {
    "frost_day": "FrostDay",
    "heat_day": "HeatDay",
    "severe_heat_day": "SevereHeatDay",
    "dry_day": "DryDay",
    "strong_wind_day": "StrongWindDay",
    "dust_storm_risk": "DustStormRisk",
    "rainy_day": "RainyDay",
}

# 准备数据
record_data = []
for row in rows:
    date = row["date"].strip()
    if not date:
        continue
    parts = date.split("-")
    record_data.append({
        "date": date,
        "maxTemp": row["temperature_2m_max"].strip(),
        "minTemp": row["temperature_2m_min"].strip(),
        "precipitation": row["precipitation_sum"].strip(),
        "windSpeed": row["wind_speed_10m_max"].strip(),
        "humidity": row["relative_humidity_2m_mean"].strip(),
        "dryStreak": row.get("dry_streak", "0").strip(),
        "rainStreak": row.get("continuous_rain_streak", "0").strip(),
        "month": f"{parts[0]}-{parts[1]}" if len(parts) >= 2 else "",
        "year": parts[0] if len(parts) >= 1 else "",
    })

# 批量导入
query = """
UNWIND $rows AS row
CREATE (r:WeatherRecord {
    date: row.date,
    maxTemp: toFloat(row.maxTemp),
    minTemp: toFloat(row.minTemp),
    precipitation: toFloat(row.precipitation),
    windSpeed: toFloat(row.windSpeed),
    humidity: toInteger(row.humidity),
    dryStreak: toInteger(row.dryStreak),
    rainStreak: toInteger(row.rainStreak),
    month: row.month,
    year: row.year
})
"""

with driver.session() as session:
    batch_size = 500
    for i in range(0, len(record_data), batch_size):
        batch = record_data[i:i + batch_size]
        session.run(query, rows=batch)
        print(f"已导入 {min(i + batch_size, len(record_data))}/{len(record_data)} 条天气记录")

### 5.5 创建天气事件节点和关系

In [ ]:
# 创建事件节点
with driver.session() as session:
    for event_name in BOOLEAN_COLUMNS.values():
        session.run("MERGE (e:WeatherEvent {name: $name})", name=event_name)
print(f"已创建 {len(BOOLEAN_COLUMNS)} 个天气事件节点")

# 创建 RECORD -> EVENT 关系
rel_data = []
for row in rows:
    date = row["date"].strip()
    if not date:
        continue
    for bool_col, event_name in BOOLEAN_COLUMNS.items():
        if row.get(bool_col, "").strip().lower() == "true":
            rel_data.append({"date": date, "event": event_name})

rel_query = """
UNWIND $rels AS rel
MATCH (r:WeatherRecord {date: rel.date})
MATCH (e:WeatherEvent {name: rel.event})
MERGE (r)-[:HAS_EVENT]->(e)
"""

with driver.session() as session:
    for i in range(0, len(rel_data), 1000):
        session.run(rel_query, rels=rel_data[i:i + 1000])
print(f"已创建 {len(rel_data)} 条 HAS_EVENT 关系")

### 5.6 创建时间序列和月份聚合

In [ ]:
with driver.session() as session:
    # NEXT_DAY 关系
    session.run("""
        MATCH (r1:WeatherRecord), (r2:WeatherRecord)
        WHERE r2.date = toString(date(r1.date) + duration('P1D'))
        MERGE (r1)-[:NEXT_DAY]->(r2)
    """)
    print("已创建 NEXT_DAY 时间序列关系")

    # 月份聚合节点
    session.run("""
        MATCH (r:WeatherRecord)
        WITH r.month AS month,
             avg(r.maxTemp) AS avgMax,
             avg(r.minTemp) AS avgMin,
             sum(r.precipitation) AS totalPrecip,
             count(r) AS days
        MERGE (m:Month {id: month})
        SET m.avgMaxTemp = round(avgMax, 1),
            m.avgMinTemp = round(avgMin, 1),
            m.totalPrecipitation = round(totalPrecip, 1),
            m.recordCount = days
    """)
    print("已创建月份聚合节点")

    # 记录-月份关系
    session.run("""
        MATCH (r:WeatherRecord), (m:Month {id: r.month})
        MERGE (r)-[:IN_MONTH]->(m)
    """)
    print("已创建 IN_MONTH 关系")

### 5.7 导入大模型抽取的知识

将大模型抽取的实体、关系和规律也导入 Neo4j。

**数据流**：
```
weather_data.csv  ──→  1097 个 WeatherRecord 节点（全量原始数据）
knowledge_*.json  ──→  实体 + 规律节点（大模型从50行样本中归纳的高层知识）
```

**类比**：CSV 是把 1097 本书搬进图书馆；大模型是请专家读了 50 本后写了一份"读书报告"贴在墙上。

In [ ]:
def import_llm_knowledge(driver, model_key, data):
    """导入大模型抽取的知识到 Neo4j"""
    parsed = data.get("parsed")
    if not parsed:
        print(f"  {model_key}: 解析失败，跳过")
        return

    # 导入实体
    entities = parsed.get("entities", [])
    imported = 0
    for ent in entities:
        ent_type = ent.get("type", "Unknown").replace(" ", "")
        props = ent.get("properties", {})
        props["name"] = ent.get("name", ent.get("id", ""))
        props["source_model"] = data["model"]
        try:
            with driver.session() as session:
                session.run(
                    f"MERGE (e:{ent_type} {{name: $name}}) SET e += $props",
                    name=props["name"], props=props,
                )
            imported += 1
        except Exception:
            pass  # 跳过约束冲突
    print(f"  {model_key}: 导入 {imported}/{len(entities)} 个实体")

    # 导入关系
    relationships = parsed.get("relationships", [])
    rel_imported = 0
    for rel in relationships:
        rel_type = rel.get("type", "RELATED_TO").upper().replace(" ", "_")
        props = rel.get("properties", {})
        props["source_model"] = data["model"]
        try:
            with driver.session() as session:
                session.run(
                    f"MATCH (a {{name: $src}}), (b {{name: $tgt}}) "
                    f"MERGE (a)-[r:{rel_type}]->(b) SET r += $props",
                    src=rel.get("source", ""), tgt=rel.get("target", ""), props=props,
                )
            rel_imported += 1
        except Exception:
            pass
    print(f"  {model_key}: 导入 {rel_imported}/{len(relationships)} 条关系")

    # 导入规律
    rules = parsed.get("rules", [])
    for i, rule in enumerate(rules):
        rule_text = rule if isinstance(rule, str) else rule.get("description", str(rule))
        with driver.session() as session:
            session.run(
                "MERGE (rule:Rule {id: $id}) "
                "SET rule.description = $desc, rule.source = $src, rule.model = $model",
                id=f"{model_key}_rule_{i}", desc=rule_text, src=model_key, model=data["model"],
            )
    print(f"  {model_key}: 导入 {len(rules)} 条规律")

print("导入函数定义完成")

In [ ]:
# 导入所有大模型的知识
for key in results:
    print(f"\n导入 {results[key]['model']} 的知识:")
    import_llm_knowledge(driver, key, results[key])

### 5.8 查看数据库统计

In [ ]:
with driver.session() as session:
    # 节点统计
    result = session.run("MATCH (n) RETURN labels(n)[0] AS label, count(n) AS cnt ORDER BY cnt DESC")
    print("=== 节点统计 ===")
    for record in result:
        print(f"  {record['label']:<20} {record['cnt']:>6} 个")

    # 关系统计
    result = session.run("MATCH ()-[r]->() RETURN type(r) AS type, count(r) AS cnt ORDER BY cnt DESC")
    print("\n=== 关系统计 ===")
    for record in result:
        print(f"  {record['type']:<20} {record['cnt']:>6} 条")

---
## 6. Neo4j 可视化查询

以下 Cypher 查询可以直接在 Neo4j Browser（`http://localhost:7474`）中执行。

### 6.1 查看全貌

In [ ]:
# 在 Python 中执行 Cypher 查询
with driver.session() as session:
    result = session.run("""
        MATCH (n)-[r]->(m)
        RETURN labels(n)[0] AS from_type, n.name AS from_name,
               type(r) AS rel,
               labels(m)[0] AS to_type, m.name AS to_name
        LIMIT 20
    """)
    print("示例关系（前 20 条）:")
    for r in result:
        print(f"  {r['from_type']}:{r['from_name']} --[{r['rel']}]--> {r['to_type']}:{r['to_name']}")

### 6.2 查询最高温超过 35°C 的日期

In [ ]:
with driver.session() as session:
    result = session.run("""
        MATCH (r:WeatherRecord)
        WHERE r.maxTemp > 35.0
        RETURN r.date AS date, r.maxTemp AS maxTemp
        ORDER BY r.maxTemp DESC
    """)
    print("最高温超过 35°C 的日期:")
    for r in result:
        print(f"  {r['date']}  {r['maxTemp']}°C")

### 6.3 统计各类极端天气事件次数

In [ ]:
with driver.session() as session:
    result = session.run("""
        MATCH (r:WeatherRecord)-[:HAS_EVENT]->(e:WeatherEvent)
        RETURN e.name AS event, count(r) AS cnt
        ORDER BY cnt DESC
    """)
    print("各类极端天气事件发生次数:")
    for r in result:
        print(f"  {r['event']:<25} {r['cnt']:>5} 次")

### 6.4 查看大模型抽取的气象规律

In [ ]:
with driver.session() as session:
    result = session.run("""
        MATCH (rule:Rule)
        RETURN rule.model AS model, rule.description AS description
        ORDER BY rule.model
    """)
    print("大模型抽取的气象规律:")
    for r in result:
        print(f"  [{r['model']}]")
        print(f"    {r['description']}")
        print()

### 6.5 同一天出现多种极端天气的记录

In [ ]:
with driver.session() as session:
    result = session.run("""
        MATCH (r:WeatherRecord)-[:HAS_EVENT]->(e:WeatherEvent)
        WITH r, collect(e.name) AS events, count(e) AS eventCount
        WHERE eventCount >= 3
        RETURN r.date AS date, r.maxTemp AS maxTemp, r.minTemp AS minTemp,
               events, eventCount
        ORDER BY eventCount DESC
        LIMIT 10
    """)
    print("同一天出现 3 种以上极端天气的记录:")
    for r in result:
        print(f"  {r['date']}  {r['maxTemp']}°C/{r['minTemp']}°C  事件数:{r['eventCount']}  {r['events']}")

### 6.6 Neo4j Browser 可视化 Cypher 查询

在 `http://localhost:7474/browser/` 中执行以下查询：

```cypher
// 查看全貌
MATCH (n)-[r]->(m) RETURN n, r, m LIMIT 200

// 查看某一天的完整信息
MATCH (r:WeatherRecord {date: "2024-08-24"})-[rel]->(target)
RETURN r, rel, target

// 查看高温日
MATCH (r:WeatherRecord)-[:HAS_EVENT]->(e:WeatherEvent {name: "HeatDay"})
RETURN r, e ORDER BY r.maxTemp DESC

// 查看大模型规律
MATCH (rule:Rule) RETURN rule.model, rule.description
```

---
## 7. 调试过程记录

### 7.1 MiMo 模型名称错误

**问题**：`Unsupported model MiMo-V2.5-Pro`（400 错误）

**原因**：模型名称大小写不对，API 要求小写

**排查**：调用 `client.models.list()` 查询可用模型列表

**修复**：`MiMo-V2.5-Pro` → `mimo-v2.5-pro`

### 7.2 JSON 解析失败

**问题**：
- DeepSeek 输出 `"streak": 1-5`（非法 JSON 值）
- MiMo 输出过长被截断（超过 token 上限）

**修复**：
- 提升 `max_tokens` 从 4000 到 8000
- 编写多层 JSON 修复逻辑（去尾逗号、修裸值、补截断括号）

### 7.3 Neo4j 连接被拒绝

**问题**：`ConnectionRefusedError: [WinError 10061]`

**原因**：`.env` 中端口配置错误，写成了 `bolt://localhost:7474`

**修复**：`7474`（HTTP 浏览器端口）→ `7687`（Bolt 程序连接端口）

### 7.4 唯一性约束冲突

**问题**：`ConstraintError: Node already exists with date = '2023-06-22'`

**原因**：通义千问抽取的实体名与已有 WeatherRecord 节点冲突

**修复**：为导入添加 try/except，跳过冲突节点

---
## 8. 关闭连接

In [ ]:
driver.close()
print("Neo4j 连接已关闭")

---
## 9. 总结

| 环节 | 工具 | 作用 |
|------|------|------|
| 原始数据 | CSV | 存 1097 天的天气数字 |
| 本体建模 | Protege + OWL | 定义领域概念和推理规则 |
| 知识抽取 | DeepSeek / 通义千问 / MiMo | 从数据中发现规律 |
| 图数据库 | Neo4j | 存储和可视化知识图谱 |
| 查询 | Cypher / SPARQL | 按关系查询知识 |

**核心发现**：
- DeepSeek-V3 在知识抽取任务上效果最佳（14 实体 + 14 关系 + 8 规律）
- 大模型抽取的知识与原始数据互补：CSV 给数字，大模型给规律
- Neo4j 图数据库天然适合展示这种"实体-关系"结构的知识